In [5]:
!pip install lightgbm catboost scikit-learn pandas numpy scipy -q



[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [6]:
# KNU 2026 ML Final Assignment - p5
# =============================================================================
# Changes from p4
# =============================================================================
# 1. Expand autocorrelation to ALL body-frame channels:
#    ayb, axb, gxb, gyb (4 new channels on top of p4's amag, azb, gmag)
#    ayb autocorr captures jog/squat cadence; axb captures lunge asymmetry.
#
# 2. Add ratio/interaction features between the top-10 features identified
#    in p4 feature importance:
#    - gyb_fft_dom_amp / gy_iqr              (normalized rotation freq amplitude)
#    - ayb_fft_band_low / ayb_fft_band_mid   (frequency profile ratio)
#    - posture_tilt_angle * amag_std          (posture + activity combined)
#    - gxb_fft_band_mid / gmag_fft_band_mid  (lateral rotation proportion)
#    - azb_acorr_l_best_lag_s * ayb_fft_dom_amp (vertical periodicity × forward energy)
#    - gyb_fft_spectral_entropy / (amag_fft_spectral_entropy + 1e-6) (rotation vs accel chaos)
#
# 3. Class-weighted training for both CatBoost and LightGBM.
#    Rest has 2.5× more samples than other classes. Upweighting minority
#    active classes should improve Squat, Lunge, JumpSquat recall at the
#    cost of a small Rest recall drop — the correct trade for BA.
#
# 4. Two-stage Rest/Active pipeline:
#    Stage 1: binary classifier (Active=0 vs Rest=1) — aggressive threshold
#    Stage 2: 8-class classifier among rows predicted Active
#    Final predictions assembled: if Stage1 → Rest, output 8; else Stage2 output.
#
# 5. Extend FFT context window from ±4s to ±6s for better low-frequency
#    resolution (0.077 Hz vs 0.111 Hz), better separating slow exercises.
#
# Data loading, device normalization, direction correction, and all other
# feature helpers are kept identical to p4.
# =============================================================================

# !pip install lightgbm catboost scikit-learn pandas numpy scipy -q

import numpy as np
import pandas as pd
from scipy import stats
from scipy.signal import find_peaks
from scipy.fft import rfft, rfftfreq
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score
import lightgbm as lgb
import catboost as cb
import os


In [7]:
# ── Configuration ──────────────────────────────────────────────────────────────
DATA_DIR = '/Users/dayana/git repo/machine-learning-class/lab6/knu-2026-machine-learning-final-assignment'
SEED     = 42
N_FOLDS  = 5

# p5: Keep LGBM params identical to p4 (p2-style slow LR).
# class_weight is handled via sample_weight per fold (see training loop).
LGBM_PARAMS = {
    'objective':         'multiclass',
    'num_class':         9,
    'metric':            'multi_logloss',
    'learning_rate':     0.05,
    'num_leaves':        127,
    'max_depth':         -1,
    'min_child_samples': 20,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      1,
    'lambda_l1':         0.1,
    'lambda_l2':         0.1,
    'verbose':           -1,
    'n_jobs':            -1,
    'seed':              SEED,
}

# p5: CatBoost — same depth/iterations as p4, add class_weights via fit().
CAT_PARAMS = {
    'iterations':             4000,
    'learning_rate':          0.05,
    'depth':                  10,
    'l2_leaf_reg':            3,
    'loss_function':          'MultiClass',
    'eval_metric':            'TotalF1:average=Macro',
    'early_stopping_rounds':  200,
    'random_seed':            SEED,
    'verbose':                0,
    'allow_writing_files':    False,
}

# p5: Two-stage binary classifier params (Active vs Rest).
# Lighter model — we only need a reliable Rest/Active split.
CAT_BINARY_PARAMS = {
    'iterations':             2000,
    'learning_rate':          0.05,
    'depth':                  8,
    'l2_leaf_reg':            3,
    'loss_function':          'Logloss',
    'eval_metric':            'AUC',
    'early_stopping_rounds':  150,
    'random_seed':            SEED,
    'verbose':                0,
    'allow_writing_files':    False,
}

LGBM_BINARY_PARAMS = {
    'objective':         'binary',
    'metric':            'binary_logloss',
    'learning_rate':     0.05,
    'num_leaves':        63,
    'max_depth':         -1,
    'min_child_samples': 20,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      1,
    'lambda_l1':         0.1,
    'lambda_l2':         0.1,
    'verbose':           -1,
    'n_jobs':            -1,
    'seed':              SEED,
}

print("Config loaded.")


Config loaded.


In [8]:
# ── Load Raw Data ──────────────────────────────────────────────────────────────
# Identical to p4
print("Loading sensor files (this may take ~1-2 minutes)...")
train_accel = pd.read_csv(os.path.join(DATA_DIR, 'train-accel.csv'))
train_gyro  = pd.read_csv(os.path.join(DATA_DIR, 'train-gyro.csv'))
train_label = pd.read_csv(os.path.join(DATA_DIR, 'train-label.csv'))
test_accel  = pd.read_csv(os.path.join(DATA_DIR, 'test-accel.csv'))
test_gyro   = pd.read_csv(os.path.join(DATA_DIR, 'test-gyro.csv'))
test_label  = pd.read_csv(os.path.join(DATA_DIR, 'test-label.csv'))

print(f"train_accel : {train_accel.shape}")
print(f"train_gyro  : {train_gyro.shape}")
print(f"train_label : {train_label.shape}")
print(f"test_accel  : {test_accel.shape}")
print(f"test_gyro   : {test_gyro.shape}")
print(f"test_label  : {test_label.shape}")


Loading sensor files (this may take ~1-2 minutes)...
train_accel : (2428374, 7)
train_gyro  : (2433673, 7)
train_label : (38015, 4)
test_accel  : (2528310, 7)
test_gyro   : (2541831, 7)
test_label  : (39473, 4)


In [9]:
# ── Device Normalization + Unknown Device Fallback — identical to p4 ───────────
def normalize_by_device(df_train, df_test, axes=['x', 'y', 'z']):
    """
    Z-score normalize sensor axes per device using train statistics.
    Unknown/unseen test devices fall back to global train mean/std.
    """
    df_train = df_train.copy()
    df_test  = df_test.copy()
    global_stats = {}
    for ax in axes:
        global_stats[ax] = (df_train[ax].mean(), df_train[ax].std() + 1e-8)
    known_devices = set(df_train['device'].unique())
    for device in known_devices:
        tr_mask = df_train['device'] == device
        te_mask = df_test['device']  == device
        for ax in axes:
            mu  = df_train.loc[tr_mask, ax].mean()
            sig = df_train.loc[tr_mask, ax].std() + 1e-8
            df_train.loc[tr_mask, ax] = (df_train.loc[tr_mask, ax] - mu) / sig
            df_test.loc[te_mask, ax]  = (df_test.loc[te_mask, ax]  - mu) / sig
    unknown_mask = ~df_test['device'].isin(known_devices)
    if unknown_mask.sum() > 0:
        print(f"  Applying global fallback normalization to {unknown_mask.sum()} rows "
              f"with unseen device(s): {df_test.loc[unknown_mask, 'device'].unique()}")
        for ax in axes:
            mu, sig = global_stats[ax]
            df_test.loc[unknown_mask, ax] = (df_test.loc[unknown_mask, ax] - mu) / sig
    return df_train, df_test

print("Normalizing accel by device...")
train_accel, test_accel = normalize_by_device(train_accel, test_accel)
print("Normalizing gyro by device...")
train_gyro,  test_gyro  = normalize_by_device(train_gyro,  test_gyro)
print("Device normalization done.")
print(f"  Devices in train_accel: {train_accel['device'].unique()}")
print(f"  Devices in test_accel : {test_accel['device'].unique()}")


Normalizing accel by device...
  Applying global fallback normalization to 92768 rows with unseen device(s): <StringArray>
['unknown']
Length: 1, dtype: str
Normalizing gyro by device...
  Applying global fallback normalization to 92737 rows with unseen device(s): <StringArray>
['unknown']
Length: 1, dtype: str
Device normalization done.
  Devices in train_accel: <StringArray>
['samsung', 'Apple']
Length: 2, dtype: str
  Devices in test_accel : <StringArray>
['unknown', 'samsung', 'Apple']
Length: 3, dtype: str


In [10]:
# ── Direction-Corrected Axes — identical to p4 ────────────────────────────────
def apply_direction_correction(df):
    """
    Rotate x, y, z into canonical body frame based on phone direction.
    Adds columns: xb (lateral), yb (forward/back), zb (vertical), magb.
    """
    df = df.copy()
    df['xb'] = df['x'].copy()
    df['yb'] = df['y'].copy()
    df['zb'] = df['z'].copy()
    for d, (sx, sy, sz) in {
        1: ( 'y', '-x',  'z'),
        2: ('-y',  'x',  'z'),
        3: ( 'y', '-x', '-z'),
        4: ('-y',  'x', '-z'),
    }.items():
        mask = df['direction'] == d
        df.loc[mask, 'xb'] = (df.loc[mask, 'x']  if sx == 'x'
                         else -df.loc[mask, 'x'] if sx == '-x'
                         else  df.loc[mask, 'y']  if sx == 'y'
                         else -df.loc[mask, 'y'])
        df.loc[mask, 'yb'] = (df.loc[mask, 'x']  if sy == 'x'
                         else -df.loc[mask, 'x'] if sy == '-x'
                         else  df.loc[mask, 'y']  if sy == 'y'
                         else -df.loc[mask, 'y'])
        df.loc[mask, 'zb'] = (df.loc[mask, 'z'] if sz == 'z'
                         else -df.loc[mask, 'z'])
    df['magb'] = np.sqrt(df['xb']**2 + df['yb']**2 + df['zb']**2)
    return df

print("Applying direction correction to accel...")
train_accel = apply_direction_correction(train_accel)
test_accel  = apply_direction_correction(test_accel)
print("Applying direction correction to gyro...")
train_gyro  = apply_direction_correction(train_gyro)
test_gyro   = apply_direction_correction(test_gyro)
print("Direction correction done.")


Applying direction correction to accel...
Applying direction correction to gyro...
Direction correction done.


In [11]:
# ── Feature Extraction Helpers ────────────────────────────────────────────────
FS = 50.0  # nominal sampling rate in Hz

# STAT FEATURES — unchanged from p2/p4 (17 per channel)
def stat_features(arr, prefix):
    feats = {}
    n = len(arr)
    if n == 0:
        for k in ['mean','std','var','min','max','range','median',
                  'p25','p75','iqr','skew','kurt','rms','energy',
                  'zero_cross','mean_abs_diff','max_abs']:
            feats[f'{prefix}_{k}'] = 0.0
        return feats
    feats[f'{prefix}_mean']          = np.mean(arr)
    feats[f'{prefix}_std']           = np.std(arr)
    feats[f'{prefix}_var']           = np.var(arr)
    feats[f'{prefix}_min']           = np.min(arr)
    feats[f'{prefix}_max']           = np.max(arr)
    feats[f'{prefix}_range']         = np.ptp(arr)
    feats[f'{prefix}_median']        = np.median(arr)
    feats[f'{prefix}_p25']           = np.percentile(arr, 25)
    feats[f'{prefix}_p75']           = np.percentile(arr, 75)
    feats[f'{prefix}_iqr']           = np.percentile(arr, 75) - np.percentile(arr, 25)
    feats[f'{prefix}_skew']          = float(stats.skew(arr))        if n > 2 else 0.0
    feats[f'{prefix}_kurt']          = float(stats.kurtosis(arr))    if n > 2 else 0.0
    feats[f'{prefix}_rms']           = np.sqrt(np.mean(arr**2))
    feats[f'{prefix}_energy']        = np.sum(arr**2) / n
    feats[f'{prefix}_zero_cross']    = np.sum(np.diff(np.sign(arr - np.mean(arr))) != 0)
    feats[f'{prefix}_mean_abs_diff'] = np.mean(np.abs(np.diff(arr))) if n > 1 else 0.0
    feats[f'{prefix}_max_abs']       = np.max(np.abs(arr))
    return feats

# FFT FEATURES — unchanged from p2/p4 (7 per channel)
def fft_features(arr, prefix, fs=FS):
    feats = {}
    n = len(arr)
    if n < 8:
        for k in ['dom_freq','dom_amp','spectral_entropy',
                  'band_low','band_mid','band_high','band_ratio_mid_low']:
            feats[f'{prefix}_{k}'] = 0.0
        return feats
    arr_centered = arr - np.mean(arr)
    fft_vals     = np.abs(rfft(arr_centered))
    freqs        = rfftfreq(n, d=1.0/fs)
    total_power  = np.sum(fft_vals**2) + 1e-10
    dom_idx      = np.argmax(fft_vals)
    feats[f'{prefix}_dom_freq']  = freqs[dom_idx]
    feats[f'{prefix}_dom_amp']   = fft_vals[dom_idx]
    psd_norm = fft_vals**2 / total_power
    psd_norm = psd_norm[psd_norm > 0]
    feats[f'{prefix}_spectral_entropy'] = -np.sum(psd_norm * np.log(psd_norm))
    low_mask  = (freqs >= 0.0) & (freqs < 1.0)
    mid_mask  = (freqs >= 1.0) & (freqs < 3.0)
    high_mask = (freqs >= 3.0) & (freqs < 8.0)
    feats[f'{prefix}_band_low']  = np.sum(fft_vals[low_mask]**2)  / total_power
    feats[f'{prefix}_band_mid']  = np.sum(fft_vals[mid_mask]**2)  / total_power
    feats[f'{prefix}_band_high'] = np.sum(fft_vals[high_mask]**2) / total_power
    feats[f'{prefix}_band_ratio_mid_low'] = (
        feats[f'{prefix}_band_mid'] / (feats[f'{prefix}_band_low'] + 1e-10)
    )
    return feats

# PEAK FEATURES — unchanged from p2/p4 (6 per channel)
def peak_features(arr, prefix, fs=FS):
    feats = {}
    n = len(arr)
    if n < 4:
        for k in ['n_peaks','peak_rate','mean_peak_height',
                  'std_peak_height','mean_peak_interval','peak_regularity']:
            feats[f'{prefix}_{k}'] = 0.0
        return feats
    arr_centered   = arr - np.mean(arr)
    height_thresh  = 0.3 * np.std(arr_centered)
    peaks, props   = find_peaks(arr_centered, height=height_thresh,
                                distance=int(fs * 0.15))
    n_peaks        = len(peaks)
    duration_s     = n / fs
    feats[f'{prefix}_n_peaks']            = n_peaks
    feats[f'{prefix}_peak_rate']          = n_peaks / duration_s
    if n_peaks > 0:
        ph = props['peak_heights']
        feats[f'{prefix}_mean_peak_height'] = np.mean(ph)
        feats[f'{prefix}_std_peak_height']  = np.std(ph)
    else:
        feats[f'{prefix}_mean_peak_height'] = 0.0
        feats[f'{prefix}_std_peak_height']  = 0.0
    if n_peaks > 1:
        intervals = np.diff(peaks) / fs
        feats[f'{prefix}_mean_peak_interval'] = np.mean(intervals)
        feats[f'{prefix}_peak_regularity']    = 1.0 / (np.std(intervals) + 1e-8)
    else:
        feats[f'{prefix}_mean_peak_interval'] = 0.0
        feats[f'{prefix}_peak_regularity']    = 0.0
    return feats

# POSTURE FEATURES — unchanged from p2/p4 (5 features)
def posture_features(a_xb, a_yb, a_zb):
    feats = {}
    if len(a_xb) == 0:
        feats.update({
            'posture_zb_mean': 0.0, 'posture_yb_mean': 0.0,
            'posture_tilt_angle': 0.0, 'posture_prone_score': 0.0,
            'posture_vertical_energy': 0.0,
        })
        return feats
    grav_x   = np.mean(a_xb)
    grav_y   = np.mean(a_yb)
    grav_z   = np.mean(a_zb)
    grav_mag = np.sqrt(grav_x**2 + grav_y**2 + grav_z**2) + 1e-8
    feats['posture_zb_mean']          = grav_z
    feats['posture_yb_mean']          = grav_y
    feats['posture_tilt_angle']       = np.arccos(np.clip(abs(grav_y) / grav_mag, -1, 1))
    feats['posture_prone_score']      = abs(grav_z) / grav_mag
    dynamic_zb = np.array(a_zb) - grav_z
    feats['posture_vertical_energy']  = np.mean(dynamic_zb**2)
    return feats

# JERK FEATURES — unchanged from p4 (13 per channel)
def jerk_features(arr, prefix, fs=FS):
    feats = {}
    keys = ['mean_abs','std','max_abs','rms','energy',
            'p90_abs','p95_abs','skew','kurt',
            'n_peaks','peak_rate','mean_peak_height','impulse_ratio']
    if len(arr) < 2:
        for k in keys: feats[f'{prefix}_{k}'] = 0.0
        return feats
    jerk = np.diff(arr.astype(np.float64)) * fs
    n    = len(jerk)
    feats[f'{prefix}_mean_abs']  = np.mean(np.abs(jerk))
    feats[f'{prefix}_std']       = np.std(jerk)
    feats[f'{prefix}_max_abs']   = np.max(np.abs(jerk))
    feats[f'{prefix}_rms']       = np.sqrt(np.mean(jerk**2))
    feats[f'{prefix}_energy']    = np.sum(jerk**2) / n
    feats[f'{prefix}_p90_abs']   = np.percentile(np.abs(jerk), 90)
    feats[f'{prefix}_p95_abs']   = np.percentile(np.abs(jerk), 95)
    feats[f'{prefix}_skew']      = float(stats.skew(jerk))       if n > 2 else 0.0
    feats[f'{prefix}_kurt']      = float(stats.kurtosis(jerk))   if n > 2 else 0.0
    abs_jerk   = np.abs(jerk)
    thresh     = 0.5 * np.std(abs_jerk)
    peaks, props = find_peaks(abs_jerk, height=thresh, distance=int(fs * 0.1))
    n_peaks    = len(peaks)
    duration_s = n / fs
    feats[f'{prefix}_n_peaks']          = n_peaks
    feats[f'{prefix}_peak_rate']        = n_peaks / duration_s
    feats[f'{prefix}_mean_peak_height'] = (
        np.mean(props['peak_heights']) if n_peaks > 0 else 0.0
    )
    mean_abs = feats[f'{prefix}_mean_abs'] + 1e-10
    feats[f'{prefix}_impulse_ratio'] = feats[f'{prefix}_max_abs'] / mean_abs
    return feats

# LATERAL ASYMMETRY FEATURES — unchanged from p4 (6 per window)
def lateral_asymmetry_features(axb, prefix='lat'):
    feats = {}
    keys  = ['xb_skew','xb_kurt','xb_asymmetry_idx',
             'xb_pos_ratio','xb_range_ratio','xb_mean_abs']
    arr   = np.asarray(axb, dtype=np.float64)
    arr   = arr[np.isfinite(arr)]
    if len(arr) < 4:
        for k in keys: feats[f'{prefix}_{k}'] = 0.0
        return feats
    arr_c = arr - np.mean(arr)
    feats[f'{prefix}_xb_skew']         = float(stats.skew(arr_c))       if len(arr_c) > 2 else 0.0
    feats[f'{prefix}_xb_kurt']         = float(stats.kurtosis(arr_c))   if len(arr_c) > 2 else 0.0
    feats[f'{prefix}_xb_asymmetry_idx']= float(abs(np.mean(arr)) / (np.std(arr) + 1e-8))
    pos_vals = arr[arr > 0]
    feats[f'{prefix}_xb_pos_ratio']    = float(len(pos_vals) / len(arr))
    feats[f'{prefix}_xb_range_ratio']  = float(np.max(arr_c) / (abs(np.min(arr_c)) + 1e-8))
    feats[f'{prefix}_xb_mean_abs']     = float(np.mean(np.abs(arr_c)))
    return feats

# AUTOCORRELATION FEATURES — p4 version (5 lags + summary, per channel)
def autocorr_features(arr, prefix, fs=FS, lags_s=(0.5, 1.0, 1.5, 2.0, 2.5)):
    feats    = {}
    lag_names= [str(lag).replace('.', 'p') for lag in lags_s]
    keys     = [f'lag_{name}' for name in lag_names] + ['max','mean','std','best_lag_s']
    arr      = np.asarray(arr, dtype=np.float64)
    arr      = arr[np.isfinite(arr)]
    if len(arr) < int(max(lags_s) * fs) + 2 or np.std(arr) < 1e-8:
        for k in keys: feats[f'{prefix}_{k}'] = 0.0
        return feats
    x    = arr - np.mean(arr)
    x    = x / (np.std(x) + 1e-8)
    vals = []
    for lag_s, lag_name in zip(lags_s, lag_names):
        lag = int(round(lag_s * fs))
        if lag <= 0 or len(x) <= lag:
            ac = 0.0
        else:
            ac = float(np.mean(x[:-lag] * x[lag:]))
        feats[f'{prefix}_lag_{lag_name}'] = ac
        vals.append(ac)
    vals = np.asarray(vals, dtype=np.float64)
    feats[f'{prefix}_max']         = float(np.max(vals))
    feats[f'{prefix}_mean']        = float(np.mean(vals))
    feats[f'{prefix}_std']         = float(np.std(vals))
    feats[f'{prefix}_best_lag_s']  = float(lags_s[int(np.argmax(vals))])
    return feats

print("Feature extraction helpers defined.")
print(f"  stat_features        : 17 per channel (unchanged)")
print(f"  fft_features         : 7 per channel (unchanged)")
print(f"  peak_features        : 6 per channel (unchanged)")
print(f"  posture_features     : 5 (unchanged)")
print(f"  jerk_features        : 13 per selected channel (unchanged)")
print(f"  lateral_asymmetry    : 6 per window (unchanged)")
print(f"  autocorr_features    : 9 channels now (p4 had 3) — NEW p5")


Feature extraction helpers defined.
  stat_features        : 17 per channel (unchanged)
  fft_features         : 7 per channel (unchanged)
  peak_features        : 6 per channel (unchanged)
  posture_features     : 5 (unchanged)
  jerk_features        : 13 per selected channel (unchanged)
  lateral_asymmetry    : 6 per window (unchanged)
  autocorr_features    : 9 channels now (p4 had 3) — NEW p5


In [12]:
# ── Main Feature Extraction — p5 ──────────────────────────────────────────────
CONTEXT_SHORT    = 1   # ±1s for stat features (unchanged)
CONTEXT_LONG     = 4   # ±4s for FFT short / jerk / autocorr / peaks (unchanged)
CONTEXT_FFT_LONG = 6   # p5 NEW: ±6s context for the extended FFT features only

def compute_features_p5(accel_df, gyro_df, label_df):
    """
    p5 feature extraction.
    Inherits all p4 features plus:
      - Autocorrelation on ayb_l, axb_l, gxb_l, gyb_l (4 new channels)
      - Extended FFT (±6s) on amag, ayb, azb, gmag, gyb for better low-freq resolution
      - Ratio/interaction features from top-10 p4 importances
    Per-PID rest-baseline normalization stays REMOVED (same as p4).
    """
    accel_df = accel_df.copy()
    gyro_df  = gyro_df.copy()
    label_df = label_df.copy()

    accel_df['time_s'] = accel_df['time'].astype(int)
    gyro_df['time_s']  = gyro_df['time'].astype(int)
    label_df['time_s'] = label_df['time'].astype(int)

    for df in [accel_df, gyro_df]:
        df['mag'] = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)

    print("  Building sensor lookup tables...")

    def build_lookup(df, cols):
        lookup = {}
        for (pid, ts), grp in df.groupby(['pid', 'time_s']):
            lookup[(pid, ts)] = grp[cols].values
        return lookup

    accel_raw_lookup  = build_lookup(accel_df, ['x','y','z','mag'])
    accel_body_lookup = build_lookup(accel_df, ['xb','yb','zb','magb'])
    gyro_raw_lookup   = build_lookup(gyro_df,  ['x','y','z','mag'])
    gyro_body_lookup  = build_lookup(gyro_df,  ['xb','yb','zb','magb'])

    meta_lookup = {}
    for (pid, ts), grp in accel_df.groupby(['pid', 'time_s']):
        meta_lookup[(pid, ts)] = {
            'direction': grp['direction'].iloc[0],
            'device':    1 if str(grp['device'].iloc[0]).lower() == 'apple' else 0,
        }

    print("  Extracting features per label row (this takes a few minutes)...")
    all_feats = []

    for idx, (_, row) in enumerate(label_df.iterrows()):
        pid = row['pid']
        ts  = int(row['time_s'])
        feat = {}

        # ── Collect SHORT window arrays (±1s) ──────────────────────────
        ax_s, ay_s, az_s, amag_s   = [], [], [], []
        gx_s, gy_s, gz_s, gmag_s   = [], [], [], []
        axb_s, ayb_s, azb_s        = [], [], []

        for offset in range(-CONTEXT_SHORT, CONTEXT_SHORT + 1):
            key = (pid, ts + offset)
            if key in accel_raw_lookup:
                c = accel_raw_lookup[key]
                ax_s.extend(c[:,0]); ay_s.extend(c[:,1])
                az_s.extend(c[:,2]); amag_s.extend(c[:,3])
            if key in accel_body_lookup:
                c = accel_body_lookup[key]
                axb_s.extend(c[:,0]); ayb_s.extend(c[:,1]); azb_s.extend(c[:,2])
            if key in gyro_raw_lookup:
                c = gyro_raw_lookup[key]
                gx_s.extend(c[:,0]); gy_s.extend(c[:,1])
                gz_s.extend(c[:,2]); gmag_s.extend(c[:,3])

        # ── Collect LONG window arrays (±4s) — for FFT short, jerk, autocorr ─
        amag_l, axb_l, ayb_l, azb_l = [], [], [], []
        gmag_l, gxb_l, gyb_l, gzb_l = [], [], [], []

        for offset in range(-CONTEXT_LONG, CONTEXT_LONG + 1):
            key = (pid, ts + offset)
            if key in accel_raw_lookup:
                c = accel_raw_lookup[key]; amag_l.extend(c[:,3])
            if key in accel_body_lookup:
                c = accel_body_lookup[key]
                axb_l.extend(c[:,0]); ayb_l.extend(c[:,1]); azb_l.extend(c[:,2])
            if key in gyro_raw_lookup:
                c = gyro_raw_lookup[key]; gmag_l.extend(c[:,3])
            if key in gyro_body_lookup:
                c = gyro_body_lookup[key]
                gxb_l.extend(c[:,0]); gyb_l.extend(c[:,1]); gzb_l.extend(c[:,2])

        # ── p5 NEW: Collect EXTENDED LONG window (±6s) for better FFT res ──
        amag_xl, ayb_xl, azb_xl = [], [], []
        gmag_xl, gyb_xl         = [], []

        for offset in range(-CONTEXT_FFT_LONG, CONTEXT_FFT_LONG + 1):
            key = (pid, ts + offset)
            if key in accel_raw_lookup:
                c = accel_raw_lookup[key]; amag_xl.extend(c[:,3])
            if key in accel_body_lookup:
                c = accel_body_lookup[key]
                ayb_xl.extend(c[:,1]); azb_xl.extend(c[:,2])
            if key in gyro_raw_lookup:
                c = gyro_raw_lookup[key]; gmag_xl.extend(c[:,3])
            if key in gyro_body_lookup:
                c = gyro_body_lookup[key]; gyb_xl.extend(c[:,1])

        # numpy conversions
        ax_s   = np.array(ax_s,   dtype=np.float32)
        ay_s   = np.array(ay_s,   dtype=np.float32)
        az_s   = np.array(az_s,   dtype=np.float32)
        amag_s = np.array(amag_s, dtype=np.float32)
        gx_s   = np.array(gx_s,   dtype=np.float32)
        gy_s   = np.array(gy_s,   dtype=np.float32)
        gz_s   = np.array(gz_s,   dtype=np.float32)
        gmag_s = np.array(gmag_s, dtype=np.float32)
        axb_s  = np.array(axb_s,  dtype=np.float32)
        ayb_s  = np.array(ayb_s,  dtype=np.float32)
        azb_s  = np.array(azb_s,  dtype=np.float32)

        amag_l = np.array(amag_l, dtype=np.float32)
        axb_l  = np.array(axb_l,  dtype=np.float32)
        ayb_l  = np.array(ayb_l,  dtype=np.float32)
        azb_l  = np.array(azb_l,  dtype=np.float32)
        gmag_l = np.array(gmag_l, dtype=np.float32)
        gxb_l  = np.array(gxb_l,  dtype=np.float32)
        gyb_l  = np.array(gyb_l,  dtype=np.float32)
        gzb_l  = np.array(gzb_l,  dtype=np.float32)

        amag_xl = np.array(amag_xl, dtype=np.float32)
        ayb_xl  = np.array(ayb_xl,  dtype=np.float32)
        azb_xl  = np.array(azb_xl,  dtype=np.float32)
        gmag_xl = np.array(gmag_xl, dtype=np.float32)
        gyb_xl  = np.array(gyb_xl,  dtype=np.float32)

        # ════════════════════════════════════════════════════════════════
        # STAT FEATURES (short window ±1s) — unchanged from p4
        # ════════════════════════════════════════════════════════════════
        feat.update(stat_features(ax_s,   'ax'))
        feat.update(stat_features(ay_s,   'ay'))
        feat.update(stat_features(az_s,   'az'))
        feat.update(stat_features(amag_s, 'amag'))
        feat.update(stat_features(axb_s,  'axb'))
        feat.update(stat_features(ayb_s,  'ayb'))
        feat.update(stat_features(azb_s,  'azb'))
        feat.update(stat_features(gx_s,   'gx'))
        feat.update(stat_features(gy_s,   'gy'))
        feat.update(stat_features(gz_s,   'gz'))
        feat.update(stat_features(gmag_s, 'gmag'))

        # Cross-sensor correlation
        if len(amag_s) > 2 and len(gmag_s) > 2:
            ml = min(len(amag_s), len(gmag_s))
            feat['accel_gyro_mag_corr'] = float(np.corrcoef(amag_s[:ml], gmag_s[:ml])[0,1])
        else:
            feat['accel_gyro_mag_corr'] = 0.0

        # ════════════════════════════════════════════════════════════════
        # FFT FEATURES (long window ±4s) — unchanged from p4
        # ════════════════════════════════════════════════════════════════
        feat.update(fft_features(amag_l, 'amag_fft'))
        feat.update(fft_features(axb_l,  'axb_fft'))
        feat.update(fft_features(ayb_l,  'ayb_fft'))
        feat.update(fft_features(azb_l,  'azb_fft'))
        feat.update(fft_features(gmag_l, 'gmag_fft'))
        feat.update(fft_features(gxb_l,  'gxb_fft'))
        feat.update(fft_features(gyb_l,  'gyb_fft'))
        feat.update(fft_features(gzb_l,  'gzb_fft'))

        # ════════════════════════════════════════════════════════════════
        # p5 NEW: EXTENDED FFT FEATURES (long window ±6s)
        # Prefix _xfft to distinguish from ±4s _fft features.
        # Better low-frequency resolution: 0.077 Hz vs 0.111 Hz.
        # Computed only on the 5 most informative channels from p4.
        # ════════════════════════════════════════════════════════════════
        feat.update(fft_features(amag_xl, 'amag_xfft'))
        feat.update(fft_features(ayb_xl,  'ayb_xfft'))
        feat.update(fft_features(azb_xl,  'azb_xfft'))
        feat.update(fft_features(gmag_xl, 'gmag_xfft'))
        feat.update(fft_features(gyb_xl,  'gyb_xfft'))

        # ════════════════════════════════════════════════════════════════
        # PEAK FEATURES (long window ±4s) — unchanged from p4
        # ════════════════════════════════════════════════════════════════
        feat.update(peak_features(amag_l, 'amag_pk'))
        feat.update(peak_features(axb_l,  'axb_pk'))
        feat.update(peak_features(ayb_l,  'ayb_pk'))
        feat.update(peak_features(azb_l,  'azb_pk'))
        feat.update(peak_features(gmag_l, 'gmag_pk'))

        # ════════════════════════════════════════════════════════════════
        # POSTURE FEATURES (body-frame short window) — unchanged
        # ════════════════════════════════════════════════════════════════
        feat.update(posture_features(axb_s, ayb_s, azb_s))

        # ════════════════════════════════════════════════════════════════
        # JERK FEATURES (targeted channels) — unchanged from p4
        # ════════════════════════════════════════════════════════════════
        feat.update(jerk_features(amag_s, 'amag_jerk_s'))
        feat.update(jerk_features(azb_s,  'azb_jerk_s'))
        feat.update(jerk_features(gmag_s, 'gmag_jerk_s'))
        feat.update(jerk_features(amag_l, 'amag_jerk_l'))
        feat.update(jerk_features(azb_l,  'azb_jerk_l'))
        feat.update(jerk_features(gmag_l, 'gmag_jerk_l'))

        # ════════════════════════════════════════════════════════════════
        # AUTOCORRELATION FEATURES — p5 expands to 7 channels total
        # p4 had: amag_l, azb_l, gmag_l
        # p5 adds: ayb_l (jog/squat cadence), axb_l (lunge asymmetry),
        #          gxb_l (lateral rotation), gyb_l (forward rotation)
        # ════════════════════════════════════════════════════════════════
        feat.update(autocorr_features(amag_l, 'amag_acorr_l'))   # p4
        feat.update(autocorr_features(azb_l,  'azb_acorr_l'))    # p4
        feat.update(autocorr_features(gmag_l, 'gmag_acorr_l'))   # p4
        feat.update(autocorr_features(ayb_l,  'ayb_acorr_l'))    # NEW p5
        feat.update(autocorr_features(axb_l,  'axb_acorr_l'))    # NEW p5
        feat.update(autocorr_features(gxb_l,  'gxb_acorr_l'))    # NEW p5
        feat.update(autocorr_features(gyb_l,  'gyb_acorr_l'))    # NEW p5

        # ════════════════════════════════════════════════════════════════
        # LATERAL ASYMMETRY FEATURES — unchanged from p4
        # ════════════════════════════════════════════════════════════════
        feat.update(lateral_asymmetry_features(axb_s, prefix='lat_s'))
        feat.update(lateral_asymmetry_features(axb_l, prefix='lat_l'))

        # ════════════════════════════════════════════════════════════════
        # GRAVITY + TILT — unchanged from p4
        # ════════════════════════════════════════════════════════════════
        if len(ax_s) > 0:
            feat['gravity_x']   = float(np.mean(ax_s))
            feat['gravity_y']   = float(np.mean(ay_s))
            feat['gravity_z']   = float(np.mean(az_s))
            gm = np.sqrt(feat['gravity_x']**2 + feat['gravity_y']**2 + feat['gravity_z']**2)
            feat['gravity_mag'] = gm
            feat['tilt_xz']     = float(np.arctan2(feat['gravity_x'], feat['gravity_z']))
            feat['tilt_yz']     = float(np.arctan2(feat['gravity_y'], feat['gravity_z']))
        else:
            feat.update({'gravity_x':0,'gravity_y':0,'gravity_z':0,
                         'gravity_mag':0,'tilt_xz':0,'tilt_yz':0})

        # Data quality
        feat['n_accel_samples'] = len(ax_s)
        feat['n_gyro_samples']  = len(gx_s)

        # Metadata
        if (pid, ts) in meta_lookup:
            feat['direction'] = meta_lookup[(pid, ts)]['direction']
            feat['device']    = meta_lookup[(pid, ts)]['device']
        else:
            feat['direction'] = -1
            feat['device']    = -1

        all_feats.append(feat)
        if (idx + 1) % 5000 == 0:
            print(f"    {idx+1}/{len(label_df)} rows done...")

    feat_df       = pd.DataFrame(all_feats)
    feat_df.index = label_df.index
    return feat_df

print("p5 feature extraction function defined.")


p5 feature extraction function defined.


In [13]:
# ── Run Feature Extraction ─────────────────────────────────────────────────────
print("Extracting TRAIN features...")
train_feats = compute_features_p5(train_accel, train_gyro, train_label)
print(f"Train features shape: {train_feats.shape}")

print("\nExtracting TEST features...")
test_feats = compute_features_p5(test_accel, test_gyro, test_label)
print(f"Test features shape: {test_feats.shape}")


Extracting TRAIN features...
  Building sensor lookup tables...
  Extracting features per label row (this takes a few minutes)...
    5000/38015 rows done...
    10000/38015 rows done...
    15000/38015 rows done...
    20000/38015 rows done...
    25000/38015 rows done...
    30000/38015 rows done...
    35000/38015 rows done...
Train features shape: (38015, 477)

Extracting TEST features...
  Building sensor lookup tables...
  Extracting features per label row (this takes a few minutes)...
    5000/39473 rows done...
    10000/39473 rows done...
    15000/39473 rows done...
    20000/39473 rows done...
    25000/39473 rows done...
    30000/39473 rows done...
    35000/39473 rows done...
Test features shape: (39473, 477)


In [14]:
# ── Remove Per-PID Rest-Baseline Normalization — same as p4 ──────────────────
train_feats_clean = train_feats.copy()
test_feats_clean  = test_feats.copy()
print("Per-PID rest-baseline feature normalization: REMOVED (same as p4).")


Per-PID rest-baseline feature normalization: REMOVED (same as p4).


In [15]:
# ── p5 NEW: Ratio / Interaction Features ──────────────────────────────────────
# Added AFTER base extraction so they reference base feature values by name.
# These encode the key nonlinear combinations that tree splits can't easily learn.

def add_interaction_features(df):
    df = df.copy()
    eps = 1e-10

    # 1. Normalized rotation frequency amplitude:
    #    High for vigorous rotation-dominated exercises (JumpSquat, Burpee),
    #    low for quasi-static poses (Rest, Squat plateau).
    df['inter_gyb_dom_amp_over_gy_iqr'] = (
        df['gyb_fft_dom_amp'] / (df['gy_iqr'] + eps)
    )

    # 2. Frequency profile ratio — low band dominant = slow exercise (Squat/Lunge);
    #    mid band dominant = jogging / jumping jacks.
    df['inter_ayb_band_low_over_mid'] = (
        df['ayb_fft_band_low'] / (df['ayb_fft_band_mid'] + eps)
    )

    # 3. Combined posture + activity: separates prone+active (MtnClimb) from
    #    prone+still (PushUp) and upright+active (Jog).
    df['inter_tilt_x_amag_std'] = (
        df['posture_tilt_angle'] * df['amag_std']
    )

    # 4. Lateral rotation proportion: captures lunge's lateral rotation vs
    #    squat's sagittal rotation.
    df['inter_gxb_mid_over_gmag_mid'] = (
        df['gxb_fft_band_mid'] / (df['gmag_fft_band_mid'] + eps)
    )

    # 5. Vertical periodicity × forward energy: JumpSquat has both strong
    #    vertical autocorrelation AND high forward-axis energy.
    df['inter_azb_acorr_x_ayb_dom_amp'] = (
        df['azb_acorr_l_best_lag_s'] * df['ayb_fft_dom_amp']
    )

    # 6. Rotation chaos over acceleration chaos: Rest has high entropy in both,
    #    but rhythmic exercises show low rotation entropy relative to accel entropy.
    df['inter_gyb_entropy_over_amag_entropy'] = (
        df['gyb_fft_spectral_entropy'] / (df['amag_fft_spectral_entropy'] + eps)
    )

    # 7. p5 bonus — autocorr cadence ratio: ayb autocorr max / azb autocorr max.
    #    Jog is strong in ayb (fore-aft); JumpSquat strong in azb (vertical).
    df['inter_ayb_acorr_over_azb_acorr'] = (
        df['ayb_acorr_l_max'] / (df['azb_acorr_l_max'] + eps)
    )

    # 8. Extended vs standard FFT band ratio — captures information gain from
    #    higher frequency resolution (±6s vs ±4s window).
    df['inter_ayb_xfft_low_over_fft_low'] = (
        df['ayb_xfft_band_low'] / (df['ayb_fft_band_low'] + eps)
    )

    return df

print("Adding interaction features to train...")
train_feats_clean = add_interaction_features(train_feats_clean)
print("Adding interaction features to test...")
test_feats_clean  = add_interaction_features(test_feats_clean)
print(f"Train features after interactions: {train_feats_clean.shape}")
print(f"Test  features after interactions: {test_feats_clean.shape}")


Adding interaction features to train...
Adding interaction features to test...
Train features after interactions: (38015, 485)
Test  features after interactions: (39473, 485)


In [16]:
# ── Prepare ML Inputs ─────────────────────────────────────────────────────────
train_feats_model = train_feats_clean.fillna(0).replace([np.inf, -np.inf], 0)
test_feats_model  = test_feats_clean.fillna(0).replace([np.inf, -np.inf],  0)

feat_cols         = list(train_feats_model.columns)
test_feats_model  = test_feats_model.reindex(columns=feat_cols, fill_value=0)

X_train = train_feats_model.values.astype(np.float32)
y_train = train_label['workout'].values
groups  = train_label['pid'].values
train_times = train_label['time'].astype(int).values

X_test      = test_feats_model.values.astype(np.float32)
test_times  = test_label['time'].astype(int).values

pid71_mask       = (test_label['pid'] == '71').values
has_sensor_mask  = ~pid71_mask

print(f"Feature columns : {len(feat_cols)}")
print(f"X_train shape   : {X_train.shape}")
print(f"X_test shape    : {X_test.shape}")
print(f"PID-71 rows     : {pid71_mask.sum()}")
print(f"Rows with sensor: {has_sensor_mask.sum()}")


Feature columns : 485
X_train shape   : (38015, 485)
X_test shape    : (39473, 485)
PID-71 rows     : 1124
Rows with sensor: 38349


In [17]:
# ── p5: Compute class weights for imbalance correction ────────────────────────
# Rest has ~2.5x more samples than active classes. We weight each class
# inversely proportional to its frequency so BA-relevant classes get equal
# gradient attention during training.
from collections import Counter

class_counts  = Counter(y_train)
n_total       = len(y_train)
n_classes     = 9
class_weights = {
    c: n_total / (n_classes * class_counts[c])
    for c in range(n_classes)
}
sample_weights_train = np.array([class_weights[y] for y in y_train], dtype=np.float32)

print("\nClass weights (inverse freq):")
for c in range(n_classes):
    print(f"  Class {c}: count={class_counts[c]:5d}  weight={class_weights[c]:.4f}")



Class weights (inverse freq):
  Class 0: count= 3389  weight=1.2464
  Class 1: count= 3523  weight=1.1989
  Class 2: count= 3575  weight=1.1815
  Class 3: count= 3697  weight=1.1425
  Class 4: count= 3610  weight=1.1701
  Class 5: count= 3627  weight=1.1646
  Class 6: count= 3589  weight=1.1769
  Class 7: count= 3795  weight=1.1130
  Class 8: count= 9210  weight=0.4586


In [18]:
# ── STAGE 1: Binary Rest/Active classifiers ───────────────────────────────────
# Labels: 0 = Active (classes 0-7), 1 = Rest (class 8)
y_binary = (y_train == 8).astype(int)

# Binary class weights: upweight Active since Rest contaminates active predictions
rest_ratio   = class_counts[8] / n_total
active_ratio = 1.0 - rest_ratio
# Slightly aggressive: downweight Rest further to push boundary toward Active
binary_weights_map = {0: 1.0 / active_ratio, 1: 0.7 / rest_ratio}
sample_weights_binary = np.array(
    [binary_weights_map[y] for y in y_binary], dtype=np.float32
)
# Normalize
sample_weights_binary /= sample_weights_binary.mean()

gkf = GroupKFold(n_splits=N_FOLDS)

print("\n" + "="*60)
print("STAGE 1: Binary Active/Rest classifiers")
print("="*60)

oof_binary_cat  = np.zeros(len(X_train))  # P(Rest) from CatBoost
oof_binary_lgbm = np.zeros(len(X_train))  # P(Rest) from LGBM
pred_binary_cat  = np.zeros(len(X_test))
pred_binary_lgbm = np.zeros(len(X_test))

binary_cat_models  = []
binary_lgbm_models = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    X_tr, X_val   = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val   = y_binary[tr_idx], y_binary[val_idx]
    sw_tr         = sample_weights_binary[tr_idx]

    # CatBoost binary
    cat_model = cb.CatBoostClassifier(**CAT_BINARY_PARAMS)
    cat_model.fit(X_tr, y_tr, sample_weight=sw_tr,
                  eval_set=(X_val, y_val), use_best_model=True, verbose=False)
    oof_binary_cat[val_idx]   = cat_model.predict_proba(X_val)[:, 1]
    pred_binary_cat           += cat_model.predict_proba(X_test)[:, 1] / N_FOLDS

    # LGBM binary
    sw_tr_lgbm = sample_weights_binary[tr_idx]
    dtrain = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr_lgbm, feature_name=feat_cols)
    dval   = lgb.Dataset(X_val, label=y_val, feature_name=feat_cols, reference=dtrain)
    lgbm_model = lgb.train(
        LGBM_BINARY_PARAMS, dtrain,
        num_boost_round=2000,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )
    oof_binary_lgbm[val_idx]   = lgbm_model.predict(X_val, num_iteration=lgbm_model.best_iteration)
    pred_binary_lgbm           += lgbm_model.predict(X_test, num_iteration=lgbm_model.best_iteration) / N_FOLDS

    # Evaluate this fold (AUC-like check via threshold 0.5)
    val_pred_rest = ((0.5 * oof_binary_cat[val_idx] + 0.5 * oof_binary_lgbm[val_idx]) > 0.5).astype(int)
    rest_recall   = np.mean(val_pred_rest[y_val==1] == 1) if (y_val==1).sum() > 0 else 0
    active_recall = np.mean(val_pred_rest[y_val==0] == 0) if (y_val==0).sum() > 0 else 0
    print(f"  Fold {fold+1} | Rest recall={rest_recall:.4f} | Active recall={active_recall:.4f} "
          f"| iter cat={cat_model.best_iteration_} lgbm={lgbm_model.best_iteration}")

    binary_cat_models.append(cat_model)
    binary_lgbm_models.append(lgbm_model)

# Blend binary predictions (equal weight for now — can tune later)
oof_binary_blend  = 0.5 * oof_binary_cat  + 0.5 * oof_binary_lgbm
pred_binary_blend = 0.5 * pred_binary_cat + 0.5 * pred_binary_lgbm

# Find best threshold for Stage 1 using OOF
# We want to tune Rest/Active split to maximize BA across all 9 classes.
# Proxy: maximize Active recall (class 0-7 aggregate recall) with minimal Rest drop.
print("\nTuning binary threshold on OOF...")
best_thresh = 0.5
best_proxy  = -1
for thresh in np.arange(0.3, 0.8, 0.02):
    pred_rest = (oof_binary_blend > thresh).astype(int)
    # Active BA proxy: recall on Active correctly called Active
    active_mask = y_binary == 0
    rest_mask   = y_binary == 1
    active_rec  = np.mean(pred_rest[active_mask] == 0) if active_mask.sum() > 0 else 0
    rest_rec    = np.mean(pred_rest[rest_mask]   == 1) if rest_mask.sum()   > 0 else 0
    # Weighted proxy: 8 active classes vs 1 rest class — weight active more
    proxy = (8 * active_rec + rest_rec) / 9
    if proxy > best_proxy:
        best_proxy  = proxy
        best_thresh = thresh

print(f"  Best binary threshold: {best_thresh:.2f}  (proxy={best_proxy:.4f})")

# Binary masks for OOF and test
oof_is_active   = (oof_binary_blend  <= best_thresh)   # True = predicted Active
test_is_active  = (pred_binary_blend <= best_thresh)    # True = predicted Active

print(f"  OOF  → Active: {oof_is_active.sum():5d}  Rest: {(~oof_is_active).sum():5d}")
print(f"  Test → Active: {test_is_active.sum():5d}  Rest: {(~test_is_active).sum():5d}")



STAGE 1: Binary Active/Rest classifiers
  Fold 1 | Rest recall=0.9729 | Active recall=0.9388 | iter cat=383 lgbm=73
  Fold 2 | Rest recall=0.9513 | Active recall=0.9916 | iter cat=421 lgbm=223
  Fold 3 | Rest recall=0.8986 | Active recall=0.9618 | iter cat=690 lgbm=83
  Fold 4 | Rest recall=0.9766 | Active recall=0.9798 | iter cat=13 lgbm=113
  Fold 5 | Rest recall=0.9548 | Active recall=0.9493 | iter cat=798 lgbm=125

Tuning binary threshold on OOF...
  Best binary threshold: 0.74  (proxy=0.9671)
  OOF  → Active: 29001  Rest:  9014
  Test → Active: 32225  Rest:  7248


In [19]:
# ── STAGE 2: 9-Class classifier (trained on full data, applied to Active rows) ─
# We still train on all 9 classes (so Stage 2 can handle any input) but the
# two-stage pipeline only uses Stage 2 predictions for rows Stage 1 calls Active.
print("\n" + "="*60)
print("STAGE 2: 9-class classifier (with class weights)")
print("="*60)

oof_lgbm  = np.zeros((len(X_train), 9))
pred_lgbm = np.zeros((len(X_test),  9))
lgbm_models = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    sw_tr       = sample_weights_train[tr_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr, feature_name=feat_cols)
    dval   = lgb.Dataset(X_val, label=y_val, feature_name=feat_cols, reference=dtrain)

    model = lgb.train(
        LGBM_PARAMS,
        dtrain,
        num_boost_round=3000,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )
    oof_lgbm[val_idx]  = model.predict(X_val, num_iteration=model.best_iteration)
    pred_lgbm         += model.predict(X_test, num_iteration=model.best_iteration) / N_FOLDS

    score = balanced_accuracy_score(y_val, np.argmax(oof_lgbm[val_idx], axis=1))
    held_pids = list(np.unique(groups[val_idx]))
    print(f"  Fold {fold+1} | PIDs: {held_pids} | BA: {score:.5f} | iter: {model.best_iteration}")
    lgbm_models.append(model)

lgbm_oof_ba = balanced_accuracy_score(y_train, np.argmax(oof_lgbm, axis=1))
print(f"\nLGBM OOF BA (raw, no Stage1 override): {lgbm_oof_ba:.5f}")

print("\n" + "="*60)
print("STAGE 2: CatBoost 9-class (with class weights)")
print("="*60)

oof_cat  = np.zeros((len(X_train), 9))
pred_cat = np.zeros((len(X_test),  9))
cat_models = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    sw_tr       = sample_weights_train[tr_idx]

    model = cb.CatBoostClassifier(**CAT_PARAMS)
    model.fit(X_tr, y_tr, sample_weight=sw_tr,
              eval_set=(X_val, y_val), use_best_model=True, verbose=False)

    oof_cat[val_idx]  = model.predict_proba(X_val)
    pred_cat         += model.predict_proba(X_test) / N_FOLDS

    score = balanced_accuracy_score(y_val, np.argmax(oof_cat[val_idx], axis=1))
    held_pids = list(np.unique(groups[val_idx]))
    print(f"  Fold {fold+1} | PIDs: {held_pids} | BA: {score:.5f} | iter: {model.best_iteration_}")
    cat_models.append(model)

cat_oof_ba = balanced_accuracy_score(y_train, np.argmax(oof_cat, axis=1))
print(f"\nCatBoost OOF BA (raw, no Stage1 override): {cat_oof_ba:.5f}")



STAGE 2: 9-class classifier (with class weights)
  Fold 1 | PIDs: ['9XOO', 'CDQ6', 'LIUY', 'MEM8', 'P3LG', 'QJ18', 'QZJ2', 'RQFN', 'TF0Y', 'UQSU', 'ZF6S'] | BA: 0.88286 | iter: 47
  Fold 2 | PIDs: ['10ZQ', '70N8', '7FRZ', '7KPX', 'CQ2G', 'DT5C', 'NQRB', 'OL6N', 'SNG7', 'TPQI', 'VBMP', 'Y21H'] | BA: 0.94988 | iter: 158
  Fold 3 | PIDs: ['2Q0J', '3C4K', '3H2A', '4UC1', 'BFIE', 'HDS9', 'IKYW', 'N6RZ', 'TGQ4', 'WZDL', 'ZL3U'] | BA: 0.87831 | iter: 58
  Fold 4 | PIDs: ['01Z2', '2XO3', '9HJO', 'D1XP', 'DU2K', 'EQZH', 'EUS2', 'F1ZM', 'IYWF', 'P4DZ', 'UWUT', 'VN8D'] | BA: 0.94865 | iter: 70
  Fold 5 | PIDs: ['13P2', '222X', '43JW', '4N6K', '7PF3', '94BI', 'C8Q6', 'J2JZ', 'MADD', 'PYXQ', 'QEYR', 'SE4Q'] | BA: 0.86588 | iter: 78

LGBM OOF BA (raw, no Stage1 override): 0.90414

STAGE 2: CatBoost 9-class (with class weights)
  Fold 1 | PIDs: ['9XOO', 'CDQ6', 'LIUY', 'MEM8', 'P3LG', 'QJ18', 'QZJ2', 'RQFN', 'TF0Y', 'UQSU', 'ZF6S'] | BA: 0.89388 | iter: 463
  Fold 2 | PIDs: ['10ZQ', '70N8', '7FRZ', 

In [20]:
# ── Find best raw blend (same grid as p4) ────────────────────────────────────
BLEND_WEIGHTS = np.round(np.arange(0.0, 0.61, 0.05), 2)
blend_rows    = []

for w in BLEND_WEIGHTS:
    blend = w * oof_lgbm + (1 - w) * oof_cat
    score = balanced_accuracy_score(y_train, np.argmax(blend, axis=1))
    blend_rows.append({'lgbm_weight': float(w), 'cat_weight': float(1-w), 'raw_oof_ba': score})

blend_df     = pd.DataFrame(blend_rows).sort_values('raw_oof_ba', ascending=False)
best_raw_w   = float(blend_df.iloc[0]['lgbm_weight'])
best_raw_score = float(blend_df.iloc[0]['raw_oof_ba'])

print("\nModel OOF BA (raw blend, no Stage1):")
print(f"  LGBM only : {lgbm_oof_ba:.5f}")
print(f"  Cat only  : {cat_oof_ba:.5f}")
print("\nTop raw blend weights:")
print(blend_df.head(8).to_string(index=False))
print(f"\nBest raw LGBM weight: {best_raw_w:.2f} | raw blend OOF BA: {best_raw_score:.5f}")



Model OOF BA (raw blend, no Stage1):
  LGBM only : 0.90414
  Cat only  : 0.91194

Top raw blend weights:
 lgbm_weight  cat_weight  raw_oof_ba
        0.30        0.70    0.913073
        0.20        0.80    0.912889
        0.25        0.75    0.912802
        0.45        0.55    0.912689
        0.40        0.60    0.912516
        0.15        0.85    0.912473
        0.35        0.65    0.912311
        0.50        0.50    0.912237

Best raw LGBM weight: 0.30 | raw blend OOF BA: 0.91307


In [21]:
# ── Soft Probability Smoothing — identical to p4 ──────────────────────────────
def soft_prob_smooth(prob_matrix, pids, times, window=5):
    prob_matrix = np.asarray(prob_matrix)
    pids        = np.asarray(pids)
    times       = np.asarray(times)
    smoothed    = prob_matrix.copy()
    hw          = window // 2
    for pid in np.unique(pids):
        idx         = np.where(pids == pid)[0]
        if len(idx) < window:
            continue
        ordered_idx  = idx[np.argsort(times[idx], kind='mergesort')]
        local_probs  = prob_matrix[ordered_idx]
        local_smooth = np.zeros_like(local_probs)
        for i in range(len(local_probs)):
            lo = max(0, i - hw)
            hi = min(len(local_probs), i + hw + 1)
            local_smooth[i] = local_probs[lo:hi].mean(axis=0)
        smoothed[ordered_idx] = local_smooth
    return smoothed


In [22]:
# ── Stage-1 override: replace Rest predictions ───────────────────────────────
def apply_stage1_override(probs_9class, is_active_mask):
    """
    For rows Stage1 calls Rest (is_active_mask=False),
    override the 9-class probs to hard Rest (class 8).
    For rows Stage1 calls Active, keep 9-class probs but zero out class 8
    and renormalize so the model focuses on distinguishing active exercises.
    """
    probs = probs_9class.copy()
    # Rows predicted Rest → hard assign class 8
    rest_rows = ~is_active_mask
    probs[rest_rows, :] = 0.0
    probs[rest_rows, 8] = 1.0
    # Rows predicted Active → suppress Rest probability, renormalize
    active_rows = is_active_mask
    probs[active_rows, 8] = 0.0
    row_sums = probs[active_rows].sum(axis=1, keepdims=True) + 1e-10
    probs[active_rows] = probs[active_rows] / row_sums
    return probs


In [23]:
# ── Evaluate blends with smoothing and Stage1 override ───────────────────────
smooth_rows = []
for w in BLEND_WEIGHTS:
    raw_probs_9   = w * oof_lgbm + (1 - w) * oof_cat

    # Version A: raw blend, no Stage1
    score_raw     = balanced_accuracy_score(y_train, np.argmax(raw_probs_9, axis=1))

    # Version B: smoothed blend, no Stage1
    smooth_probs  = soft_prob_smooth(raw_probs_9, groups, train_times, window=5)
    score_smooth  = balanced_accuracy_score(y_train, np.argmax(smooth_probs, axis=1))

    # Version C: Stage1 override, then smooth
    s1_probs      = apply_stage1_override(raw_probs_9.copy(), oof_is_active)
    s1_smooth     = soft_prob_smooth(s1_probs, groups, train_times, window=5)
    score_s1      = balanced_accuracy_score(y_train, np.argmax(s1_smooth, axis=1))

    best_score    = max(score_raw, score_smooth, score_s1)
    best_mode     = ['raw','smooth','s1+smooth'][[score_raw, score_smooth, score_s1].index(best_score)]

    smooth_rows.append({
        'lgbm_weight': float(w),
        'raw_oof_ba':     score_raw,
        'smooth_oof_ba':  score_smooth,
        's1_smooth_ba':   score_s1,
        'best_oof_ba':    best_score,
        'best_mode':      best_mode,
    })

smooth_df = pd.DataFrame(smooth_rows).sort_values('best_oof_ba', ascending=False)
best      = smooth_df.iloc[0]
best_w    = float(best['lgbm_weight'])
best_mode = best['best_mode']

print("\nTop blend/smoothing/Stage1 candidates:")
print(smooth_df.head(10).to_string(index=False))
print(f"\nSelected LGBM weight: {best_w:.2f} | best mode: {best_mode}")



Top blend/smoothing/Stage1 candidates:
 lgbm_weight  raw_oof_ba  smooth_oof_ba  s1_smooth_ba  best_oof_ba best_mode
        0.45    0.912689       0.915872      0.918792     0.918792 s1+smooth
        0.50    0.912237       0.915736      0.918727     0.918727 s1+smooth
        0.55    0.911401       0.915096      0.918446     0.918446 s1+smooth
        0.35    0.912311       0.915887      0.918386     0.918386 s1+smooth
        0.40    0.912516       0.915646      0.918381     0.918381 s1+smooth
        0.30    0.913073       0.915920      0.918298     0.918298 s1+smooth
        0.60    0.911153       0.914571      0.918259     0.918259 s1+smooth
        0.25    0.912802       0.916034      0.918183     0.918183 s1+smooth
        0.20    0.912889       0.915857      0.917769     0.917769 s1+smooth
        0.15    0.912473       0.915685      0.917434     0.917434 s1+smooth

Selected LGBM weight: 0.45 | best mode: s1+smooth


In [24]:
# ── Build final OOF and test predictions ─────────────────────────────────────
oof_blend_probs  = best_w * oof_lgbm  + (1 - best_w) * oof_cat
test_blend_probs = best_w * pred_lgbm + (1 - best_w) * pred_cat

if best_mode == 'raw':
    final_oof_preds  = np.argmax(oof_blend_probs, axis=1)
    test_preds_final = np.argmax(test_blend_probs, axis=1)
elif best_mode == 'smooth':
    final_oof_preds  = np.argmax(soft_prob_smooth(oof_blend_probs,  groups,          train_times), axis=1)
    test_preds_final = np.argmax(soft_prob_smooth(test_blend_probs, test_label['pid'].values, test_times), axis=1)
else:  # s1+smooth
    s1_oof   = apply_stage1_override(oof_blend_probs.copy(),  oof_is_active)
    s1_test  = apply_stage1_override(test_blend_probs.copy(), test_is_active)
    final_oof_preds  = np.argmax(soft_prob_smooth(s1_oof,  groups,                    train_times), axis=1)
    test_preds_final = np.argmax(soft_prob_smooth(s1_test, test_label['pid'].values,  test_times),  axis=1)

final_ba = balanced_accuracy_score(y_train, final_oof_preds)
print(f"\nFinal selected OOF BA: {final_ba:.5f}")
print(f"(p4 was 0.91132, p2 was 0.90851)")



Final selected OOF BA: 0.91879
(p4 was 0.91132, p2 was 0.90851)


In [25]:
# ── Per-class recall vs p4 ────────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix

class_names = [
    '0:JumpJack','1:Jog','2:Squat','3:MtnClimb',
    '4:PushUp','5:Burpee','6:Lunge','7:JumpSquat','8:Rest'
]
p4_recalls = [0.968, 0.937, 0.857, 0.883, 0.923, 0.926, 0.848, 0.894, 0.966]

cm = confusion_matrix(y_train, final_oof_preds)
print("\nPer-class recall (p5 vs p4):")
for i, name in enumerate(class_names):
    r     = cm[i, i] / cm[i].sum() if cm[i].sum() > 0 else 0
    delta = r - p4_recalls[i]
    sign  = '+' if delta >= 0 else '-'
    print(f"  {name:20s}: {r:.3f}  {sign}{abs(delta):.3f} vs p4")



Per-class recall (p5 vs p4):
  0:JumpJack          : 0.967  -0.001 vs p4
  1:Jog               : 0.943  +0.006 vs p4
  2:Squat             : 0.882  +0.025 vs p4
  3:MtnClimb          : 0.891  +0.008 vs p4
  4:PushUp            : 0.940  +0.017 vs p4
  5:Burpee            : 0.942  +0.016 vs p4
  6:Lunge             : 0.886  +0.038 vs p4
  7:JumpSquat         : 0.902  +0.008 vs p4
  8:Rest              : 0.917  -0.049 vs p4


In [26]:
# ── Build Submission ──────────────────────────────────────────────────────────
submission = pd.read_csv(os.path.join(DATA_DIR, 'submission.csv'))
submission.loc[has_sensor_mask, 'workout'] = test_preds_final[has_sensor_mask]
submission.loc[pid71_mask,      'workout'] = 8   # PID 71: no sensor data → Rest
submission['workout'] = submission['workout'].astype(int)

print("\nSubmission preview:")
print(submission.head(10))
print("\nPrediction distribution:")
print(submission['workout'].value_counts().sort_index())



Submission preview:
      id  workout
0  56209        8
1  56210        8
2  56211        7
3  56212        7
4  56213        7
5  56214        7
6  56215        7
7  56216        7
8  56217        7
9  56218        7

Prediction distribution:
workout
0    3881
1    3546
2    3936
3    3530
4    4038
5    3894
6    4373
7    3650
8    8625
Name: count, dtype: int64


In [27]:
# ── Save Submission ───────────────────────────────────────────────────────────
out_path = os.path.join(".", 'submission_p5.csv')
submission.to_csv(out_path, index=False)
print(f"\nSaved → {out_path}")
print("Done.")



Saved → ./submission_p5.csv
Done.


In [28]:
# ── Feature Importance ────────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fi_lgbm = pd.DataFrame({
    'feature':    feat_cols,
    'importance': lgbm_models[-1].feature_importance(importance_type='gain'),
}).sort_values('importance', ascending=False)

fi_cat = pd.DataFrame({
    'feature':    feat_cols,
    'importance': cat_models[-1].get_feature_importance(),
}).sort_values('importance', ascending=False)

p5_new_tags = ['acorr', 'xfft', 'inter_']
print("\nTop 30 LGBM features:")
print(fi_lgbm.head(30).to_string(index=False))
print(f"\np5 new features in LGBM top 30: "
      f"{[f for f in fi_lgbm.head(30)['feature'] if any(t in f for t in p5_new_tags)]}")

print("\nTop 30 CatBoost features:")
print(fi_cat.head(30).to_string(index=False))
print(f"\np5 new features in CatBoost top 30: "
      f"{[f for f in fi_cat.head(30)['feature'] if any(t in f for t in p5_new_tags)]}")

fig, ax = plt.subplots(figsize=(10, 12))
plt.barh(fi_cat['feature'].head(30)[::-1], fi_cat['importance'].head(30)[::-1])
plt.xlabel('Feature importance')
plt.title(f'Top 30 Feature Importances - p5 CatBoost (OOF BA={final_ba:.5f})')
plt.tight_layout()
plt.savefig('feature_importance_p5.png', dpi=100)
plt.show()
print("Feature importance chart saved → feature_importance_p5.png")



Top 30 LGBM features:
                            feature   importance
                   ayb_fft_band_low 73910.141157
                         ax_max_abs 68621.364858
                             gy_iqr 68011.676351
                             gy_rms 43833.748902
                   gxb_fft_band_mid 43121.175959
inter_gyb_entropy_over_amag_entropy 39643.901508
                    gyb_fft_dom_amp 38865.122393
                    ayb_fft_dom_amp 34681.521757
             ayb_pk_std_peak_height 29352.466975
                   gxb_fft_dom_freq 28268.355696
                   gyb_xfft_dom_amp 26955.587986
                accel_gyro_mag_corr 26791.978690
                 posture_tilt_angle 19487.082631
                   gxb_fft_band_low 17392.038354
                             ax_std 16860.463987
      inter_gyb_dom_amp_over_gy_iqr 16329.255256
                             ax_rms 15302.102136
                            ayb_std 15038.641000
                             ax_var 12380.9015